### Question 1: CPython Cache and local variables (function scope)

In [4]:
# If a variable value is same then python might intern string that means one single str object and ob_refcnt multiple.

def f():
    x = "Hello"
    return id(x)            # return address of object "Hello" where x is currently pointing [x------>"Hello"]

def g():
    x = "Hello"
    return id(x)            # return address of object "Hello" where x is currently pointing [x------>"Hello"]

print(f"{f()=}, {g()=}")

print(f() is g())
print(f() == g())

f()=2269741373008, g()=2269741373008
False
True


In [3]:
# CPython mini-integer-cache: -5-256. Hence both functions local variables are pointing to single integer object 5. 

def f():
    x = 5
    return id(x)            # return address of integer object 5 where x is currently pointing [x------>5]

def g():
    x = 5
    return id(x)            # return address of integer object 5 where x is currently pointing [x------>5]

print(f"{f()=}, {g()=}")

print(f() is g())           
print(f() == g())

f()=140726461629688, g()=140726461629688
False
True


**So why does is return False?** Because of what your function is returning. You are returning the id()—which is a completely different object! Let's break down exactly what happens in system memory during your print(f() is g()) statement.

#### The Breakdown

##### Step 1: The execution of f():

1. x = 5: Python points x to the cached integer 5 at memory address 140726461629688.
2. id(x): Python grabs that memory address.
3. return id(x): Your function is now returning the integer value 140726461629688.
4. The Catch: Because 140726461629688 is way, way larger than the CPython integer cache limit of 256, Python must create a brand new integer object in memory just to hold this giant number. Let's say it places this giant number object at address 0x999.

##### Step 2: The execution of g():

1. x = 5: Python points x to the same cached integer 5 at 140726461629688.
2. id(x): Python grabs the address.
3. return id(x): Your function returns 140726461629688.
4. The Catch (again): Python must create another brand new integer object to hold this giant number. Let's say it places this second giant number object at address 0x888.

#### Step 3: The Comparison:

When you run print(f() is g()), here is what Python is actually comparing under the hood:

- Object at 0x999 is Object at 0x888

Even though both of these large integer objects contain the exact same numeric value inside them (140726461629688), they are two completely different physical objects in memory because they were generated dynamically and exceed the cache limit of 256.

Therefore:

- is returns False (Address 0x999 != Address 0x888)
- == returns True (Value 140726461629688 == Value 140726461629688)

### Let's prove it

In [9]:
def f():
    x = 5
    return x   # Return the actual cached object

def g():
    x = 5
    return x   # Return the actual cached object

print(f() is g())

True


Output True! Because now, you are comparing the cached object 5 directly against the cached object 5

In [10]:
print(id(f()) == id(g()))

True


##### Breakdown of print(id(f()) == id(g()))

This is famously known as the "overlapping lifetimes" illusion. Here is the exact, step-by-step timeline of what happens inside your CPU when you execute print(id(f()) == id(g())).

Python evaluates the == operator from left to right.

##### Step 1: Evaluating the Left Side id(f())

1. Python calls f(). It returns a massive integer object holding the address of 5. Let's say Python creates this giant integer object in memory at address 0xABC.
2. Python calls id() on that object. id() looks at the object, reads its address, and evaluates to the number 0xABC.
3. The Magic Moment (Garbage Collection): As soon as id() finishes extracting that number, the massive integer object created by f() has served its purpose. Absolutely nothing is pointing to it anymore. Its reference count (refcnt) instantly drops to 0.
4. The Garbage Collector swoops in, instantly destroys the object, and tells the Operating System: "Hey, memory address 0xABC is vacant again!"

##### Step 2: Evaluating the Right Side id(g())

1. Now Python moves to the right side of the == and calls g().
2. g() returns another massive integer object. Python needs a place in memory to put it.
3. The Memory Allocator (Recycling): Python's memory allocator is highly efficient. It looks at the memory pool and says, "Wait, a perfectly sized chunk of memory at 0xABC was literally just freed up a microsecond ago. Let's just put the new object right there!"
4. The second massive integer object is created at 0xABC.
5. Python calls id() on it, which evaluates to the number 0xABC.

##### Step 3: The Final Comparison

By the time the == operator actually fires, the temporary objects are gone, and Python is simply left comparing the two resulting numbers:

`0xABC == 0xABC`

Result: `True`

##### The Big Takeaway

proved that two completely different objects can have the exact same id()... as long as they don't exist at the exact same time.

The first object was born, lived, and died at 0xABC. Then, the second object was born, lived, and died at 0xABC. Because their lifetimes never overlapped, they safely shared the exact same physical memory address sequentially.

This is a notorious Python "footgun." It’s exactly why experienced Python developers will tell you: Never use id(a) == id(b) to check if two things are the same object. Always use a is b, because is keeps both objects alive simultaneously during the comparison, preventing the Garbage Collector from pulling this recycling trick!

> Above explanations were explained by Gemini. I have verified outputs and resolve my queries, so if there is any mistake then it can do or might I have overlooked anything while writing notes so if there is any then please fix it. Thanks!

<br>

---

<br>

### Question 2

In [8]:
x = 10
y = 10

print(f"{id(x)=}, {id(y)=}")

print(x is y)
print(x == y)

id(x)=140726461629848, id(y)=140726461629848
True
True
